<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/2026Fall/Inventory_Replenishment_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Synthetic Sales Data

Because a real sales dataset was not provided, this project uses synthetic data generated with Python. The dataset contains 90 days of daily demand for three SKUs. Each SKU has a different average demand level, and random variation is included to make the data more realistic for forecasting.

In [25]:
# Generate sales.csv

import numpy as np
import pandas as pd

# Set a seed so the same data is created every time
rng = np.random.default_rng(7)

# Create 90 days of sales dates
dates = pd.date_range(
    start="2025-01-01",
    periods=90,
    freq="D"
)

# Use three different products
skus = ["SKU001", "SKU002", "SKU003"]

# Average daily demand for each SKU
average_demand = {
    "SKU001": 20,
    "SKU002": 12,
    "SKU003": 8
}

sales_rows = []

for sku in skus:
    for date in dates:

        # Weekly demand pattern
        weekly_effect = 3 * np.sin(
            2 * np.pi * date.dayofweek / 7
        )

        # Random demand variation
        random_effect = rng.normal(0, 3)

        # Make sure demand is never negative
        qty_sold = max(
            0,
            round(
                average_demand[sku]
                + weekly_effect
                + random_effect
            )
        )

        sales_rows.append({
            "date": date,
            "sku": sku,
            "qty_sold": qty_sold
        })

# Convert the list into a DataFrame
sales = pd.DataFrame(sales_rows)

# Save the data as a CSV file
sales.to_csv("sales.csv", index=False)

# Display the first few rows
print("sales.csv was created successfully.")
display(sales.head(10))

sales.csv was created successfully.


,date,sku,qty_sold
0,2025-01-01,SKU001,23
1,2025-01-02,SKU001,22
2,2025-01-03,SKU001,18
3,2025-01-04,SKU001,14
4,2025-01-05,SKU001,16
5,2025-01-06,SKU001,17
6,2025-01-07,SKU001,23
7,2025-01-08,SKU001,27
8,2025-01-09,SKU001,20
9,2025-01-10,SKU001,17


## Explanation of the Sales Data

The sales dataset contains three columns:

- `date`: The date when the product was sold.
- `sku`: The product identification number.
- `qty_sold`: The number of units sold on that date.

The data includes 90 days of sales for three products. SKU001 has the highest average demand, SKU002 has medium demand, and SKU003 has the lowest demand. A weekly demand pattern and random variation were added so the data can be used to test the EWMA forecasting model and inventory replenishment agent.

In [26]:
# Check the size and structure of the sales data

print("Number of rows:", len(sales))
print("Number of SKUs:", sales["sku"].nunique())
print("Date range:", sales["date"].min(), "to", sales["date"].max())

print("\nSales by SKU:")
display(sales.groupby("sku")["qty_sold"].agg(["count", "mean", "sum"]))

print("\nMissing values:")
display(sales.isnull().sum())

Number of rows: 270
Number of SKUs: 3
Date range: 2025-01-01 00:00:00 to 2025-03-31 00:00:00

Sales by SKU:


,count,mean,sum
sku,,,
SKU001,90,19.500000,1755
SKU002,90,11.377778,1024
SKU003,90,7.666667,690



Missing values:


,0
date,0
sku,0
qty_sold,0


In [27]:
# Save the DataFrame as a CSV file
sales.to_csv("sales.csv", index=False)

# Download the CSV file to your computer
from google.colab import files
files.download("sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Create Inventory and Policy Parameter Data

The inventory file contains the opening stock available for each SKU at the beginning of the simulation. The parameter file contains the costs and replenishment rules used by the inventory agent.

Each SKU has different starting inventory, lead time, minimum order quantity, and service-level requirements. These values allow the agent to make realistic order-or-wait decisions.

In [28]:
# Create inventory.csv

inventory = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "opening_stock": [180, 120, 80]
})

# Save the file
inventory.to_csv("inventory.csv", index=False)

print("inventory.csv was created successfully.")
display(inventory)

inventory.csv was created successfully.


,sku,opening_stock
0,SKU001,180
1,SKU002,120
2,SKU003,80


In [29]:
# Create params.csv

params = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "unit_cost": [10.00, 15.00, 20.00],
    "holding_cost_per_day": [0.20, 0.30, 0.40],
    "stockout_cost": [25.00, 30.00, 40.00],
    "lead_time_days": [5, 7, 4],
    "min_order_qty": [50, 30, 20],
    "service_level": [0.95, 0.90, 0.95]
})

# Save the file
params.to_csv("params.csv", index=False)

print("params.csv was created successfully.")
display(params)

params.csv was created successfully.


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU001,10.0,0.2,25.0,5,50,0.95
1,SKU002,15.0,0.3,30.0,7,30,0.90
2,SKU003,20.0,0.4,40.0,4,20,0.95


## Explanation of Inventory and Policy Parameters

The `inventory.csv` file contains the opening stock for each product. SKU001 starts with 180 units, SKU002 starts with 120 units, and SKU003 starts with 80 units.

The `params.csv` file contains the business rules and cost information for each SKU:

- `unit_cost`: Cost of purchasing one unit.
- `holding_cost_per_day`: Cost of storing one unit for one day.
- `stockout_cost`: Penalty for each unit of unmet demand.
- `lead_time_days`: Number of days between placing and receiving an order.
- `min_order_qty`: Smallest quantity the agent is allowed to order.
- `service_level`: Target probability of meeting customer demand.

The agent uses these values to balance customer service and inventory cost. A higher service level and stockout cost encourage the agent to keep more safety stock, while holding cost discourages unnecessary over-ordering.

In [30]:
# Verify that all required CSV files exist and have the correct columns

sales_check = pd.read_csv("sales.csv")
inventory_check = pd.read_csv("inventory.csv")
params_check = pd.read_csv("params.csv")

assert list(sales_check.columns) == ["date", "sku", "qty_sold"]

assert list(inventory_check.columns) == [
    "sku",
    "opening_stock"
]

assert list(params_check.columns) == [
    "sku",
    "unit_cost",
    "holding_cost_per_day",
    "stockout_cost",
    "lead_time_days",
    "min_order_qty",
    "service_level"
]

assert set(sales_check["sku"]) == set(inventory_check["sku"])
assert set(sales_check["sku"]) == set(params_check["sku"])

print("All three CSV files have the required format.")

All three CSV files have the required format.


In [31]:
from google.colab import files

files.download("inventory.csv")
files.download("params.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Inventory Replenishment Agent

This section loads the CSV files, creates an EWMA demand forecast, calculates safety stock, simulates inventory decisions, and compares the agent with a baseline policy.

In [32]:
from statistics import NormalDist
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the required CSV files
sales = pd.read_csv("sales.csv", parse_dates=["date"])
inventory = pd.read_csv("inventory.csv")
params = pd.read_csv("params.csv")

# Aggregate demand by date and SKU
daily_demand = (
    sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
    .sum()
)

# Forecast settings
ALPHA = 0.30
REVIEW_PERIOD_DAYS = 1

# Create a daily EWMA forecast for each SKU
forecast_data = []

for sku, group in daily_demand.groupby("sku"):
    group = group.sort_values("date").copy()

    # Use previous demand to forecast the next day
    group["forecast"] = (
        group["qty_sold"]
        .shift(1)
        .ewm(alpha=ALPHA, adjust=False)
        .mean()
    )

    # Use the first observed demand as the initial forecast
    group["forecast"] = group["forecast"].fillna(group["qty_sold"].iloc[0])

    # Forecast error and expanding error standard deviation
    group["forecast_error"] = (
        group["qty_sold"] - group["forecast"]
    )

    group["error_std"] = (
        group["forecast_error"]
        .expanding()
        .std(ddof=1)
        .fillna(0)
    )

    forecast_data.append(group)

forecast_data = pd.concat(forecast_data, ignore_index=True)

print("EWMA forecast created successfully.")
display(forecast_data.head(10))

EWMA forecast created successfully.


,date,sku,qty_sold,forecast,forecast_error,error_std
0,2025-01-01,SKU001,23,23.000000,0.000000,0.000000
1,2025-01-02,SKU001,22,23.000000,-1.000000,0.707107
2,2025-01-03,SKU001,18,22.700000,-4.700000,2.475884
3,2025-01-04,SKU001,14,21.290000,-7.290000,3.368930
4,2025-01-05,SKU001,16,19.103000,-3.103000,2.918295
5,2025-01-06,SKU001,17,18.172100,-1.172100,2.740654
6,2025-01-07,SKU001,23,17.820470,5.179530,3.941197
7,2025-01-08,SKU001,27,19.374329,7.625671,4.924118
8,2025-01-09,SKU001,20,21.662030,-1.662030,4.620782
9,2025-01-10,SKU001,17,21.163421,-4.163421,4.493606


# Inventory Replenishment Agent Simulation

In [33]:
def simulate_policy(policy_name):
    logs = []
    orders = []
    summary = []

    for sku, group in forecast_data.groupby("sku"):
        group = group.sort_values("date").reset_index(drop=True)

        item_inventory = inventory[
            inventory["sku"] == sku
        ].iloc[0]

        item_params = params[
            params["sku"] == sku
        ].iloc[0]

        on_hand = float(item_inventory["opening_stock"])

        lead_time = int(item_params["lead_time_days"])
        min_order_qty = int(item_params["min_order_qty"])
        service_level = float(item_params["service_level"])
        holding_cost_rate = float(
            item_params["holding_cost_per_day"]
        )
        stockout_cost_rate = float(
            item_params["stockout_cost"]
        )
        unit_cost = float(item_params["unit_cost"])

        pending_orders = []

        total_demand = 0
        total_fulfilled = 0
        total_lost_sales = 0
        total_holding_cost = 0
        total_stockout_cost = 0
        stockout_days = 0

        for _, row in group.iterrows():
            date = row["date"]
            demand = float(row["qty_sold"])
            beginning_inventory = on_hand

            # Receive purchase orders that have arrived
            arrivals = sum(
                order["quantity"]
                for order in pending_orders
                if order["arrival_date"] <= date
            )

            pending_orders = [
                order
                for order in pending_orders
                if order["arrival_date"] > date
            ]

            on_hand += arrivals

            # Fulfill today's demand
            fulfilled = min(on_hand, demand)
            lost_sales = max(0, demand - fulfilled)
            on_hand -= fulfilled

            if lost_sales > 0:
                stockout_days += 1

            forecast = max(0, float(row["forecast"]))
            error_std = max(0, float(row["error_std"]))

            # Safety stock formula
            z_value = NormalDist().inv_cdf(service_level)

            safety_stock = (
                z_value
                * error_std
                * math.sqrt(lead_time + REVIEW_PERIOD_DAYS)
            )

            # Inventory position includes pending orders
            inventory_position = (
                on_hand
                + sum(
                    order["quantity"]
                    for order in pending_orders
                )
            )

            if policy_name == "agent":
                projected_demand = forecast * (
                    lead_time + REVIEW_PERIOD_DAYS
                )

                target_inventory = (
                    projected_demand + safety_stock
                )

                order_gap = (
                    target_inventory - inventory_position
                )

                if order_gap > 0:
                    order_quantity = max(
                        min_order_qty,
                        math.ceil(order_gap)
                    )
                    action = "PLACE ORDER"
                    reason = (
                        "Projected inventory is below "
                        "the target level."
                    )
                else:
                    order_quantity = 0
                    action = "WAIT"
                    reason = (
                        "Inventory is sufficient for "
                        "projected demand."
                    )

            else:
                # Simple baseline: fixed reorder rule
                baseline_reorder_point = forecast * lead_time

                if inventory_position < baseline_reorder_point:
                    order_quantity = max(
                        min_order_qty,
                        math.ceil(
                            baseline_reorder_point
                            - inventory_position
                        )
                    )
                    action = "PLACE ORDER"
                    reason = (
                        "Baseline reorder point was reached."
                    )
                else:
                    order_quantity = 0
                    action = "WAIT"
                    reason = (
                        "Inventory is above the baseline "
                        "reorder point."
                    )

            # Schedule the purchase order
            if order_quantity > 0:
                arrival_date = date + pd.Timedelta(
                    days=lead_time
                )

                pending_orders.append({
                    "arrival_date": arrival_date,
                    "quantity": order_quantity
                })

                orders.append({
                    "policy": policy_name,
                    "date": date,
                    "sku": sku,
                    "order_quantity": order_quantity,
                    "arrival_date": arrival_date
                })

            daily_holding_cost = (
                on_hand * holding_cost_rate
            )

            daily_stockout_cost = (
                lost_sales * stockout_cost_rate
            )

            total_demand += demand
            total_fulfilled += fulfilled
            total_lost_sales += lost_sales
            total_holding_cost += daily_holding_cost
            total_stockout_cost += daily_stockout_cost

            logs.append({
                "policy": policy_name,
                "date": date,
                "sku": sku,
                "beginning_inventory": beginning_inventory,
                "arrivals": arrivals,
                "demand": demand,
                "fulfilled_units": fulfilled,
                "lost_sales": lost_sales,
                "ending_inventory": on_hand,
                "forecast": forecast,
                "safety_stock": safety_stock,
                "inventory_position": inventory_position,
                "order_quantity": order_quantity,
                "action": action,
                "reason": reason,
                "holding_cost": daily_holding_cost,
                "stockout_cost": daily_stockout_cost,
                "inventory_value": on_hand * unit_cost
            })

        fill_rate = (
            total_fulfilled / total_demand
            if total_demand > 0
            else 1
        )

        total_cost = (
            total_holding_cost
            + total_stockout_cost
        )

        summary.append({
            "policy": policy_name,
            "sku": sku,
            "total_demand": total_demand,
            "units_fulfilled": total_fulfilled,
            "stockout_units": total_lost_sales,
            "stockout_days": stockout_days,
            "fill_rate": fill_rate,
            "holding_cost": total_holding_cost,
            "stockout_cost": total_stockout_cost,
            "order_cost": 0,
            "total_cost": total_cost
        })

    return (
        pd.DataFrame(logs),
        pd.DataFrame(summary),
        pd.DataFrame(orders)
    )

In [34]:
agent_log, agent_summary, agent_orders = simulate_policy(
    "agent"
)

baseline_log, baseline_summary, baseline_orders = simulate_policy(
    "baseline"
)

print("Agent and baseline simulations completed.")

Agent and baseline simulations completed.


# Results and Evaluation

In [37]:
# Display the daily agent log
display(agent_log.head(20))

# Save the full decision log
agent_log.to_csv("decision_log.csv", index=False)

# Save and download the daily agent decision log
agent_log.to_csv("decision_log.csv", index=False)

from google.colab import files
files.download("decision_log.csv")

# Compare the agent and baseline by SKU
comparison = pd.concat(
    [agent_summary, baseline_summary],
    ignore_index=True
)

display(comparison)

# Calculate overall results
overall_results = comparison.groupby(
    "policy",
    as_index=False
).agg({
    "total_demand": "sum",
    "units_fulfilled": "sum",
    "stockout_units": "sum",
    "stockout_days": "sum",
    "holding_cost": "sum",
    "stockout_cost": "sum",
    "order_cost": "sum",
    "total_cost": "sum"
})

overall_results["fill_rate"] = (
    overall_results["units_fulfilled"]
    / overall_results["total_demand"]
)

print("Overall performance comparison:")
display(overall_results)

# Save the overall results as a CSV file
overall_results.to_csv("overall_results.csv", index=False)

# Download the file
from google.colab import files
files.download("overall_results.csv")

,policy,date,sku,beginning_inventory,arrivals,demand,fulfilled_units,lost_sales,ending_inventory,forecast,safety_stock,inventory_position,order_quantity,action,reason,holding_cost,stockout_cost,inventory_value
0,agent,2025-01-01,SKU001,180.0,0,23.0,23.0,0,157.0,23.000000,0.000000,157.0,0,WAIT,Inventory is sufficient for projected demand.,31.4,0.0,1570.0
1,agent,2025-01-02,SKU001,157.0,0,22.0,22.0,0,135.0,23.000000,2.848970,135.0,50,PLACE ORDER,Projected inventory is below the target level.,27.0,0.0,1350.0
2,agent,2025-01-03,SKU001,135.0,0,18.0,18.0,0,117.0,22.700000,9.975464,167.0,0,WAIT,Inventory is sufficient for projected demand.,23.4,0.0,1170.0
3,agent,2025-01-04,SKU001,117.0,0,14.0,14.0,0,103.0,21.290000,13.573596,153.0,0,WAIT,Inventory is sufficient for projected demand.,20.6,0.0,1030.0
4,agent,2025-01-05,SKU001,103.0,0,16.0,16.0,0,87.0,19.103000,11.757962,137.0,0,WAIT,Inventory is sufficient for projected demand.,17.4,0.0,870.0
5,agent,2025-01-06,SKU001,87.0,0,17.0,17.0,0,70.0,18.172100,11.042237,120.0,50,PLACE ORDER,Projected inventory is below the target level.,14.0,0.0,700.0
6,agent,2025-01-07,SKU001,70.0,50,23.0,23.0,0,97.0,17.820470,15.879288,147.0,0,WAIT,Inventory is sufficient for projected demand.,19.4,0.0,970.0
7,agent,2025-01-08,SKU001,97.0,0,27.0,27.0,0,70.0,19.374329,19.839529,120.0,50,PLACE ORDER,Projected inventory is below the target level.,14.0,0.0,700.0
8,agent,2025-01-09,SKU001,70.0,0,20.0,20.0,0,50.0,21.662030,18.617373,150.0,0,WAIT,Inventory is sufficient for projected demand.,10.0,0.0,500.0
9,agent,2025-01-10,SKU001,50.0,0,17.0,17.0,0,33.0,21.163421,18.104975,133.0,50,PLACE ORDER,Projected inventory is below the target level.,6.6,0.0,330.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,policy,sku,total_demand,units_fulfilled,stockout_units,stockout_days,fill_rate,holding_cost,stockout_cost,order_cost,total_cost
0,agent,SKU001,1755.0,1755.0,0.0,0,1.000000,1153.4,0.0,0,1153.4
1,agent,SKU002,1024.0,1024.0,0.0,0,1.000000,1156.5,0.0,0,1156.5
2,agent,SKU003,690.0,690.0,0.0,0,1.000000,1224.4,0.0,0,1224.4
3,baseline,SKU001,1755.0,1718.0,37.0,6,0.978917,604.6,925.0,0,1529.6
4,baseline,SKU002,1024.0,970.0,54.0,10,0.947266,639.6,1620.0,0,2259.6
5,baseline,SKU003,690.0,623.0,67.0,16,0.902899,588.4,2680.0,0,3268.4


Overall performance comparison:


,policy,total_demand,units_fulfilled,stockout_units,stockout_days,holding_cost,stockout_cost,order_cost,total_cost,fill_rate
0,agent,3469.0,3469.0,0.0,0,3534.3,0.0,0,3534.3,1.000000
1,baseline,3469.0,3311.0,158.0,32,1832.6,5225.0,0,7057.6,0.954454


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
import os

print(os.path.exists("/content/decision_log.csv"))

if os.path.exists("/content/decision_log.csv"):
    print("File size:", os.path.getsize("/content/decision_log.csv"), "bytes")

True
File size: 45016 bytes


## Trade-off Reflection

The agent balances customer service with inventory cost. Keeping more safety stock can improve the fill rate and reduce stockouts, but it also increases holding costs. Keeping less inventory reduces storage costs but increases the risk of unmet demand and stockout penalties. The agent provides more transparent decisions because its daily log explains whether it placed an order or waited and why.

## Scaling Note

If this inventory replenishment agent were used in a real business, it could run as a scheduled cloud job once per day. A cloud function or container could automatically load the newest sales, inventory, and parameter files, run the forecasting and replenishment process, and save the daily decision log to cloud storage or a database.

The system should monitor fill rate, stockout units, forecast error, inventory levels, order quantities, and total cost. Alerts could be sent when the fill rate falls below the target service level, forecast error increases significantly, or inventory reaches an unsafe level.

For reliability, the system should validate that required files and columns are present, identify missing sales data, handle invalid parameters, and retry failed data loads. It should also record errors so a person can review them.

To control costs, the agent should use inventory caps, minimum and maximum order limits, and a spending limit. It should track holding costs, stockout costs, and money tied up in inventory. Human approval should be required before unusually large orders are sent to a supplier.